#### Loading the datasets into the workspace

In [246]:
import pandas as pd
import numpy as np

In [247]:

df_ed = pd.read_csv("datasets/raw/ed_visits.csv")


In [248]:
df_ed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   ed_visit_id         6000 non-null   object
 1   patient_id          6000 non-null   object
 2   arrival_datetime    6000 non-null   object
 3   departure_datetime  6000 non-null   object
 4   triage_level        6000 non-null   int64 
 5   triage_category     6000 non-null   object
 6   chief_complaint     6000 non-null   object
 7   day_of_week         6000 non-null   object
 8   hour_of_arrival     6000 non-null   int64 
 9   month               6000 non-null   int64 
 10  season              6000 non-null   object
 11  wait_time_minutes   6000 non-null   int64 
 12  door_to_doctor_min  6000 non-null   int64 
 13  ed_los_minutes      6000 non-null   int64 
 14  imaging_ordered     6000 non-null   int64 
 15  labs_ordered        6000 non-null   int64 
 16  iv_access           6000

#### Correcting the data types of the columns of our dataset

In [249]:
# ================================
# Correct Data Types
# ================================

# ---------- IDs ----------
df_ed["ed_visit_id"] = df_ed["ed_visit_id"].astype("string")
df_ed["patient_id"] = df_ed["patient_id"].astype("string")

# ---------- Datetime ----------
df_ed["arrival_datetime"] = pd.to_datetime(df_ed["arrival_datetime"])
df_ed["departure_datetime"] = pd.to_datetime(df_ed["departure_datetime"])

# ---------- Ordinal / Calendar ----------
df_ed["triage_level"] = df_ed["triage_level"].astype("int8")
df_ed["hour_of_arrival"] = df_ed["hour_of_arrival"].astype("int8")
df_ed["month"] = df_ed["month"].astype("int8")

# ---------- Durations ----------
duration_cols = [
    "wait_time_minutes",
    "door_to_doctor_min",
    "ed_los_minutes"
]

df_ed[duration_cols] = df_ed[duration_cols].astype("int32")

# ---------- Binary Flags ----------
binary_cols = [
    "imaging_ordered",
    "labs_ordered",
    "iv_access",
    "admitted_from_ed",
    "left_ama"
]

df_ed[binary_cols] = df_ed[binary_cols].astype("int8")

# ---------- Categoricals ----------
categorical_cols = [
    "triage_category",
    "chief_complaint",
    "day_of_week",
    "season",
    "disposition",
    "attending_ed_md",
    "hospital"
]

for col in categorical_cols:
    df_ed[col] = df_ed[col].astype("category")

In [250]:
df_ed.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ed_visit_id         6000 non-null   string        
 1   patient_id          6000 non-null   string        
 2   arrival_datetime    6000 non-null   datetime64[ns]
 3   departure_datetime  6000 non-null   datetime64[ns]
 4   triage_level        6000 non-null   int8          
 5   triage_category     6000 non-null   category      
 6   chief_complaint     6000 non-null   category      
 7   day_of_week         6000 non-null   category      
 8   hour_of_arrival     6000 non-null   int8          
 9   month               6000 non-null   int8          
 10  season              6000 non-null   category      
 11  wait_time_minutes   6000 non-null   int32         
 12  door_to_doctor_min  6000 non-null   int32         
 13  ed_los_minutes      6000 non-null   int32       

#### Step 1 — Aggregate to Daily Level

In [251]:
df = df_ed.copy()

In [252]:
# Ensure datetime sorted
df = df.sort_values("arrival_datetime")

# Daily aggregation
daily_ed = (
    df
    .set_index("arrival_datetime")
    .resample("D")
    .agg({
        "ed_visit_id": "count",      # total arrivals
        "wait_time_minutes": "mean",
        "door_to_doctor_min": "mean",
        "ed_los_minutes": "mean",
        "admitted_from_ed": "sum",
        "left_ama": "sum",
        "imaging_ordered": "sum",
        "labs_ordered": "sum"
    })
)

daily_ed.rename(columns={"ed_visit_id": "ed_arrivals"}, inplace=True)

In [253]:
daily_ed.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1827 entries, 2020-01-01 to 2024-12-31
Freq: D
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ed_arrivals         1827 non-null   int64  
 1   wait_time_minutes   1746 non-null   float64
 2   door_to_doctor_min  1746 non-null   float64
 3   ed_los_minutes      1746 non-null   float64
 4   admitted_from_ed    1827 non-null   int8   
 5   left_ama            1827 non-null   int8   
 6   imaging_ordered     1827 non-null   int8   
 7   labs_ordered        1827 non-null   int8   
dtypes: float64(3), int64(1), int8(4)
memory usage: 78.5 KB


#### Create a continous time index

In [254]:
full_range = pd.date_range(
    start=daily_ed.index.min(),
    end=daily_ed.index.max(),
    freq="D"
)

daily_ed = daily_ed.reindex(full_range)

#### Step 2 — Feature Engineering

In [255]:
daily_ed.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1827 entries, 2020-01-01 to 2024-12-31
Freq: D
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ed_arrivals         1827 non-null   int64  
 1   wait_time_minutes   1746 non-null   float64
 2   door_to_doctor_min  1746 non-null   float64
 3   ed_los_minutes      1746 non-null   float64
 4   admitted_from_ed    1827 non-null   int8   
 5   left_ama            1827 non-null   int8   
 6   imaging_ordered     1827 non-null   int8   
 7   labs_ordered        1827 non-null   int8   
dtypes: float64(3), int64(1), int8(4)
memory usage: 78.5 KB


#### Fill missing days

In [256]:
daily_ed["ed_arrivals"] = daily_ed["ed_arrivals"].fillna(0)

daily_ed.fillna(method="ffill", inplace=True)

C:\Users\unclesteve\AppData\Local\Temp\ipykernel_9280\3138982266.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  daily_ed.fillna(method="ffill", inplace=True)


In [257]:
daily_ed.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1827 entries, 2020-01-01 to 2024-12-31
Freq: D
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ed_arrivals         1827 non-null   int64  
 1   wait_time_minutes   1827 non-null   float64
 2   door_to_doctor_min  1827 non-null   float64
 3   ed_los_minutes      1827 non-null   float64
 4   admitted_from_ed    1827 non-null   int8   
 5   left_ama            1827 non-null   int8   
 6   imaging_ordered     1827 non-null   int8   
 7   labs_ordered        1827 non-null   int8   
dtypes: float64(3), int64(1), int8(4)
memory usage: 78.5 KB


#### Time Based Feature Engineering

##### Calendar Feature

In [258]:
daily_ed["day_of_week"] = daily_ed.index.dayofweek
daily_ed["week_of_year"] = daily_ed.index.isocalendar().week
daily_ed["month"] = daily_ed.index.month
daily_ed["quarter"] = daily_ed.index.quarter
daily_ed["year"] = daily_ed.index.year

##### Weekend Indicator

In [259]:
daily_ed["is_weekend"] = (
    daily_ed["day_of_week"] >= 5
).astype(int)

##### Cyclical Encoding

In [260]:
daily_ed["dow_sin"] = np.sin(
    2*np.pi*daily_ed["day_of_week"]/7
)
daily_ed["dow_cos"] = np.cos(
    2*np.pi*daily_ed["day_of_week"]/7
)

#### Lag Feature

##### Standard NHS lags

In [261]:
for lag in [1,2,3,7,14,28]:
    daily_ed[f"arrivals_lag_{lag}"] = daily_ed["ed_arrivals"].shift(lag)

#### Rolling Statistics

In [262]:
daily_ed["rolling_mean_7"] = daily_ed["ed_arrivals"].rolling(7).mean()
daily_ed["rolling_std_7"] = daily_ed["ed_arrivals"].rolling(7).std()

daily_ed["rolling_mean_28"] = daily_ed["ed_arrivals"].rolling(28).mean()
daily_ed["rolling_std_28"] = daily_ed["ed_arrivals"].rolling(28).std()

#### Operational Preasure Features

In [263]:
daily_ed["admission_rate"] = (
    daily_ed["admitted_from_ed"] /
    daily_ed["ed_arrivals"]
)

#### Throughput Stree

In [264]:
daily_ed["throughput_pressure"] = (
    daily_ed["wait_time_minutes"] +
    daily_ed["door_to_doctor_min"]
)

#### Leakage Removal

In [265]:
#Drop early rows created by lag features
daily_ed.dropna(inplace=True)

#### Triage Level / Severity Mix

##### Daily acuity

In [266]:
# Create daily triage mix
triage_daily = (
    df
    .set_index("arrival_datetime")
    .groupby([
        pd.Grouper(freq="D"),
        "triage_category"
    ])
    .size()
    .unstack(fill_value=0)
)

triage_daily.columns = triage_daily.columns.astype(str)

# Normalize index
#triage_daily.index = triage_daily.index.normalize()
#daily_ed.index = daily_ed.index.normalize()

# Join safely
#daily_ed = daily_ed.join(triage_daily, how="left")

# Fill missing values
#daily_ed.fillna(0, inplace=True)

C:\Users\unclesteve\AppData\Local\Temp\ipykernel_9280\4256793962.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([


In [267]:
severity_weights = {
    "Resuscitation": 5,
    "Emergent": 4,
    "Urgent": 3,
    "Less Urgent": 2,
    "Non-Urgent": 1
}

In [268]:
# Copy to avoid modifying original
triage_weighted = triage_daily.copy()

for col in triage_weighted.columns:
    weight = severity_weights.get(col, 0)  # SAFE lookup
    triage_weighted[col] = triage_weighted[col] * weight

# Create severity index
daily_ed["severity_index"] = triage_weighted.sum(axis=1)

#### Crowding/congestion feature

In [269]:
daily_ed["admission_rate"] = (
    daily_ed["admitted_from_ed"] /
    daily_ed["ed_arrivals"]
)

daily_ed["avg_wait_time"] = daily_ed["wait_time_minutes"]
daily_ed["avg_los"] = daily_ed["ed_los_minutes"]

In [270]:
daily_ed["crowding_index"] = (
    daily_ed["avg_wait_time"] *
    daily_ed["admission_rate"]
)

#### System capacity Feature

In [271]:
daily_ed["boarding_proxy"] = (
    daily_ed["avg_los"] *
    daily_ed["admission_rate"]
)

#### Chief Complaints Mix

In [272]:
def safe_daily_join(base_df, new_df):

    # --- normalize index ---
    base_df.index = pd.to_datetime(base_df.index).normalize()
    new_df.index = pd.to_datetime(new_df.index).normalize()

    # --- remove categorical columns ---
    new_df.columns = new_df.columns.astype(str)

    # --- remove overlapping columns ---
    overlap = base_df.columns.intersection(new_df.columns)

    if len(overlap) > 0:
        print(f"Dropping existing columns before join: {list(overlap)}")
        base_df = base_df.drop(columns=overlap)

    # --- safe join ---
    merged = base_df.join(new_df, how="left")

    return merged.fillna(0)

In [273]:
# Calling the function
daily_ed = safe_daily_join(daily_ed, triage_daily)

#### Chief Complaints Mix

In [274]:
top_cc = df["chief_complaint"].value_counts().nlargest(5).index

cc_daily = (
    df[df["chief_complaint"].isin(top_cc)]
    .groupby([
        pd.Grouper(key="arrival_datetime", freq="D"),
        "chief_complaint"
    ])
    .size()
    .unstack(fill_value=0)
)

cc_daily.columns = cc_daily.columns.astype(str)

C:\Users\unclesteve\AppData\Local\Temp\ipykernel_9280\603652491.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([


In [275]:
# Calling the function
daily_ed = safe_daily_join(daily_ed, cc_daily)

#### Hospital (Multisite forecasting)

In [276]:
hospital_daily = (
    df.groupby([
        pd.Grouper(key="arrival_datetime", freq="D"),
        "hospital"
    ])
    .size()
    .unstack(fill_value=0)
)

hospital_daily.columns = hospital_daily.columns.astype(str)
daily_ed = daily_ed.join(hospital_daily)

C:\Users\unclesteve\AppData\Local\Temp\ipykernel_9280\1812383427.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([


#### Auto Regressive Trend

In [277]:
daily_ed["trend_7"] = (
    daily_ed["ed_arrivals"]
    .diff(7)
)

daily_ed["trend_28"] = (
    daily_ed["ed_arrivals"]
    .diff(28)
)

In [278]:
daily_ed.columns

Index(['ed_arrivals', 'wait_time_minutes', 'door_to_doctor_min',
       'ed_los_minutes', 'admitted_from_ed', 'left_ama', 'imaging_ordered',
       'labs_ordered', 'day_of_week', 'week_of_year', 'month', 'quarter',
       'year', 'is_weekend', 'dow_sin', 'dow_cos', 'arrivals_lag_1',
       'arrivals_lag_2', 'arrivals_lag_3', 'arrivals_lag_7', 'arrivals_lag_14',
       'arrivals_lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28',
       'rolling_std_28', 'admission_rate', 'throughput_pressure',
       'severity_index', 'avg_wait_time', 'avg_los', 'crowding_index',
       'boarding_proxy', 'Emergent', 'Less Urgent', 'Non-Urgent',
       'Resuscitation', 'Urgent', 'Abdominal pain', 'Altered mental status',
       'Back pain', 'Chest pain', 'Drug overdose', 'Fever', 'Headache',
       'Hypertensive urgency', 'Hypoglycemia', 'Laceration', 'Leg swelling',
       'Nausea and vomiting', 'Palpitations', 'Seizure', 'Shortness of breath',
       'Stroke symptoms', 'Syncope / fall', 'Tr

#### Remove NA from lags

In [279]:
daily_ed = daily_ed.dropna()

In [280]:
daily_ed.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1690 entries, 2020-02-26 to 2024-12-31
Data columns (total 68 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ed_arrivals            1690 non-null   int64  
 1   wait_time_minutes      1690 non-null   float64
 2   door_to_doctor_min     1690 non-null   float64
 3   ed_los_minutes         1690 non-null   float64
 4   admitted_from_ed       1690 non-null   int8   
 5   left_ama               1690 non-null   int8   
 6   imaging_ordered        1690 non-null   int8   
 7   labs_ordered           1690 non-null   int8   
 8   day_of_week            1690 non-null   int32  
 9   week_of_year           1690 non-null   UInt32 
 10  month                  1690 non-null   int32  
 11  quarter                1690 non-null   int32  
 12  year                   1690 non-null   int32  
 13  is_weekend             1690 non-null   int64  
 14  dow_sin                1690 non-null  

#### Saving the final version of the daily_ed dataset

In [285]:
daily_ed.columns = daily_ed.columns.str.replace(" ", "_")

In [286]:
# Create datasets folder if it doesn't exist
from pathlib import Path

Path("datasets").mkdir(exist_ok=True)

# Save forecasting-ready dataset
daily_ed.to_csv(
    "datasets/daily_ed_forecasting.csv",
    index=True  # IMPORTANT: keeps datetime index
)

print("✅ daily_ed_forecasting.csv saved successfully")

✅ daily_ed_forecasting.csv saved successfully
